In [ ]:
pip install -U gliner datasets accelerate seqeval sentencepiece


In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("LocalDoc/azerbaijani-ner-dataset")
print(ds)
print(ds["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 99545
    })
})
{'index': '640b71a8-014e-424b-96e1-80c74c9317bb', 'tokens': "['Komitədən', 'bildirilib', 'ki', ',', 'sovet', 'dövründə', 'Azərbaycanda', 'cəmi', '17', 'məscid', 'fəaliyyət', 'göstərirdisə', ',', 'dövlət', 'müstəqilliyinin', 'bərpasından', 'sonra', 'ölkədə', '814', 'məscid', 'tikilib', '.']", 'ner_tags': '[3, 0, 0, 0, 0, 0, 14, 0, 17, 0, 0, 0, 0, 3, 0, 0, 0, 14, 17, 0, 0, 0]'}


In [ ]:
import ast
from datasets import load_dataset, Dataset, DatasetDict

raw_ds = load_dataset("LocalDoc/azerbaijani-ner-dataset")

def safe_parse_list(x):
    if x is None:
        return None
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        return ast.literal_eval(x)
    return None

def normalize_tokens(x):
    x = safe_parse_list(x)
    if x is None:
        return None
    return [str(t) for t in x]

def normalize_tags(x):
    x = safe_parse_list(x)
    if x is None:
        return None
    try:
        return [int(t) for t in x]
    except Exception:
        return None

def convert_split(split, split_name):
    rows = []
    skipped = 0

    for i, ex in enumerate(split):
        tokens = normalize_tokens(ex.get("tokens"))
        ner_tags = normalize_tags(ex.get("ner_tags"))

        if tokens is None or ner_tags is None:
            skipped += 1
            continue

        if len(tokens) != len(ner_tags):
            skipped += 1
            continue

        rows.append({
            "index": str(ex.get("index")),
            "tokens": tokens,
            "ner_tags": ner_tags,
        })

    print(split_name, "kechilen/buraxilan setirler:", skipped)
    return Dataset.from_list(rows)

parts = {}
for split_name in raw_ds.keys():
    parts[split_name] = convert_split(raw_ds[split_name], split_name)

ds = DatasetDict(parts)

print(ds)
print(ds["train"][0])
print(type(ds["train"][0]["tokens"]))
print(type(ds["train"][0]["ner_tags"]))
print(type(ds["train"][0]["ner_tags"][0]))


train kechilen/buraxilan setirler: 3592
DatasetDict({
    train: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 95953
    })
})
{'index': '640b71a8-014e-424b-96e1-80c74c9317bb', 'tokens': ['Komitədən', 'bildirilib', 'ki', ',', 'sovet', 'dövründə', 'Azərbaycanda', 'cəmi', '17', 'məscid', 'fəaliyyət', 'göstərirdisə', ',', 'dövlət', 'müstəqilliyinin', 'bərpasından', 'sonra', 'ölkədə', '814', 'məscid', 'tikilib', '.'], 'ner_tags': [3, 0, 0, 0, 0, 0, 14, 0, 17, 0, 0, 0, 0, 3, 0, 0, 0, 14, 17, 0, 0, 0]}
<class 'list'>
<class 'list'>
<class 'int'>


In [ ]:
split1 = ds["train"].train_test_split(test_size=0.1, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

final_ds = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"],
})

print(final_ds)


DatasetDict({
    train: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 86357
    })
    validation: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 4798
    })
    test: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 4798
    })
})


In [ ]:
import json

id2label = {
    0: "O",
    1: "PERSON",
    2: "LOCATION",
    3: "ORGANISATION",
    4: "DATE",
    5: "TIME",
    6: "MONEY",
    7: "PERCENTAGE",
    8: "FACILITY",
    9: "PRODUCT",
    10: "EVENT",
    11: "ART",
    12: "LAW",
    13: "LANGUAGE",
    14: "GPE",
    15: "NORP",
    16: "ORDINAL",
    17: "CARDINAL",
    18: "DISEASE",
    19: "CONTACT",
    20: "ADAGE",
    21: "QUANTITY",
    22: "MISCELLANEOUS",
    23: "POSITION",
    24: "PROJECT",
}

def to_gliner_example(example):
    tokens = example["tokens"]
    tags = example["ner_tags"]

    ner = []
    current_label = None
    start_idx = None

    for i, tag_id in enumerate(tags):
        if tag_id == 0:
            if current_label is not None:
                ner.append([start_idx, i - 1, current_label])
                current_label = None
                start_idx = None
            continue

        label = id2label[tag_id].lower()

        if current_label is None:
            current_label = label
            start_idx = i
        elif label != current_label:
            ner.append([start_idx, i - 1, current_label])
            current_label = label
            start_idx = i

    if current_label is not None:
        ner.append([start_idx, len(tags) - 1, current_label])

    return {
        "tokenized_text": tokens,
        "ner": ner,
    }

def convert_split(split, drop_empty=True):
    rows = []
    for ex in split:
        row = to_gliner_example(ex)
        if drop_empty and len(row["ner"]) == 0:
            continue
        rows.append(row)
    return rows

train_data = convert_split(final_ds["train"], drop_empty=True)
val_data = convert_split(final_ds["validation"], drop_empty=True)
test_data = convert_split(final_ds["test"], drop_empty=False)

print(len(train_data), len(val_data), len(test_data))
print(train_data[0])

with open("train_gliner.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False)

with open("val_gliner.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False)

with open("test_gliner.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False)


65491 3605 4798
{'tokenized_text': ['Heyvan', 'sahibinə', 'külli', 'miqdarda', 'maddi', 'ziyan', 'vurulub', '.'], 'ner': [[2, 3, 'quantity']]}


In [ ]:
import gc
import torch
from gliner import GLiNER

gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

MAX_TOKENS = 96

train_data_small = [
    x for x in train_data
    if 1 <= len(x["tokenized_text"]) <= MAX_TOKENS
][:10000]

val_data_small = [
    x for x in val_data
    if 1 <= len(x["tokenized_text"]) <= MAX_TOKENS
][:1000]

print(len(train_data_small), len(val_data_small))
print(max(len(x["tokenized_text"]) for x in train_data_small))


10000 1000
65


In [ ]:
model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

model.train_model(
    train_dataset=train_data_small,
    eval_dataset=val_data_small,
    output_dir="gliner_az_model",
    max_steps=300,
    learning_rate=2e-5,
    others_lr=1e-5,
    weight_decay=0.01,
    others_weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    logging_steps=20,
    save_steps=150,
    save_total_limit=1,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 0}.


Step,Training Loss
20,8.395565
40,10.929897
60,10.308515
80,5.073404
100,5.736319
120,7.690482
140,6.157304
160,6.378675
180,7.619476
200,7.929974


In [ ]:
id2label = {
    0: "O",
    1: "PERSON",
    2: "LOCATION",
    3: "ORGANISATION",
    4: "DATE",
    5: "TIME",
    6: "MONEY",
    7: "PERCENTAGE",
    8: "FACILITY",
    9: "PRODUCT",
    10: "EVENT",
    11: "ART",
    12: "LAW",
    13: "LANGUAGE",
    14: "GPE",
    15: "NORP",
    16: "ORDINAL",
    17: "CARDINAL",
    18: "DISEASE",
    19: "CONTACT",
    20: "ADAGE",
    21: "QUANTITY",
    22: "MISCELLANEOUS",
    23: "POSITION",
    24: "PROJECT",
}


In [ ]:
from collections import Counter

label_token_counts = Counter()
label_row_counts = Counter()
broken_rows = 0

for ex in final_ds["train"]:
    tokens = ex["tokens"]
    tags = ex["ner_tags"]

    if tokens is None or tags is None or len(tokens) != len(tags):
        broken_rows += 1
        continue

    labels_in_this_row = set()

    for tag in tags:
        if tag != 0:
            label = id2label[tag]
            label_token_counts[label] += 1
            labels_in_this_row.add(label)

    for label in labels_in_this_row:
        label_row_counts[label] += 1

print("broken row sayi:", broken_rows)
print("\nToken sayina gore label statistikasi:")
for label, count in label_token_counts.most_common():
    print(label, count)

print("\nRow sayina gore label statistikasi:")
for label, count in label_row_counts.most_common():
    print(label, count)


broken row sayi: 0

Token sayina gore label statistikasi:
ORGANISATION 56811
PERSON 48843
GPE 31590
MISCELLANEOUS 30217
DATE 23591
CARDINAL 19744
LOCATION 12196
POSITION 7727
PRODUCT 7201
EVENT 7153
FACILITY 7094
ORDINAL 5459
NORP 4414
ART 3915
MONEY 3490
QUANTITY 2852
DISEASE 2551
PROJECT 2119
TIME 1922
PERCENTAGE 1185
LANGUAGE 554
LAW 513
CONTACT 442
ADAGE 235

Row sayina gore label statistikasi:
ORGANISATION 26232
PERSON 22912
GPE 21395
DATE 15522
CARDINAL 14043
LOCATION 8766
POSITION 5939
EVENT 5596
MISCELLANEOUS 5225
ORDINAL 4876
FACILITY 4721
PRODUCT 4159
NORP 3770
ART 2391
MONEY 2245
QUANTITY 2078
DISEASE 2000
PROJECT 1646
TIME 1317
PERCENTAGE 892
LANGUAGE 433
LAW 358
CONTACT 313
ADAGE 193


In [ ]:
import random

def show_example(ex):
    print("TEXT:", " ".join(ex["tokens"]))
    print("ENTITIES:")
    for token, tag in zip(ex["tokens"], ex["ner_tags"]):
        if tag != 0:
            print(f"  {token} -> {id2label[tag]}")
    print("-" * 50)

sample_ids = random.sample(range(len(final_ds["train"])), 100)

for idx in sample_ids:
    show_example(final_ds["train"][idx])


TEXT: Hətta Samanilər dövlətində seyyidlərə məxsus olan torpaqlar vergiyə tabe edilməmişdi .
ENTITIES:
  Samanilər -> ORGANISATION
--------------------------------------------------
TEXT: Naxçıvanla sərhədə nəzarət edən korpusun komandiri coğrafi baxımdan təmas xəttində hər iki tərəfin üstünlükləri olduğunu bildirib : " Cəbhənin müxtəlif istiqamətlərində vəziyyət müxtəlifdir . 

ENTITIES:
  Naxçıvanla -> LOCATION
--------------------------------------------------
TEXT: Müəyyən olunmuş müddətdə qeydiyyatdan keçməmək abituriyentin müvafiq ixtisasa yerləşdirilmədən imtina etməsi kimi qiymətləndiriləcək və onun qəbul haqqında əmri təhsil müəssisəsinə göndəriləməyəcək .
ENTITIES:
  təhsil -> ORGANISATION
--------------------------------------------------
TEXT: N . Allahverdiyev sabah Hollandiyaya yollanacaq yığmanın qrup oyunlarında uğur qazanacağının çətin olacağını bildirib : “ Səfərə ciddi itki ilə gedəcəyik və bu səbəbdən obyektiv olmaq gərəkdir .
ENTITIES:
  N -> PERSON
  Allahverdiyev

### *These examples indicate that certain labels in the dataset, particularly MISCELLANEOUS and ORGANISATION, have been applied overly broadly and, in some cases, do not accurately reflect the semantic meaning of the annotated tokens. Additionally, multi-word named entities are frequently annotated at the individual token level rather than as complete spans. This suggests that the annotation scheme has not been applied consistently across the dataset, pointing to limitations in annotation quality. Moreover, the relatively large label set appears to introduce ambiguity between closely related categories, which may increase the difficulty of model learning and negatively affect overall performance.*


In [ ]:
keep_labels = {
    "PERSON",
    "LOCATION",
    "ORGANISATION",
    "DATE",
    "TIME",
    "MONEY",
    "PRODUCT",
    "EVENT",
    "CONTACT",
    "FACILITY",
    "POSITION",
    "CARDINAL",
    "ORDINAL",
    "QUANTITY",
    "PERCENTAGE",
}

def simplify_label(label):
    if label == "GPE":
        return "LOCATION"
    if label in {"CARDINAL", "ORDINAL", "QUANTITY", "PERCENTAGE"}:
        return "NUMBER"
    if label in keep_labels:
        return label
    return "O"


In [ ]:
def convert_to_gliner(split):
    rows = []

    for ex in split:
        tokens = ex["tokens"]
        tags = ex["ner_tags"]

        ner = []
        current_label = None
        start_idx = None

        for i, tag_id in enumerate(tags):
            raw_label = id2label[tag_id]
            label = simplify_label(raw_label)

            if label == "O":
                if current_label is not None:
                    ner.append([start_idx, i - 1, current_label.lower()])
                    current_label = None
                    start_idx = None
                continue

            if current_label is None:
                current_label = label
                start_idx = i
            elif current_label != label:
                ner.append([start_idx, i - 1, current_label.lower()])
                current_label = label
                start_idx = i

        if current_label is not None:
            ner.append([start_idx, len(tags) - 1, current_label.lower()])

        if len(ner) > 0:
            rows.append({
                "tokenized_text": tokens,
                "ner": ner,
            })

    return rows


In [ ]:
train_simple = convert_to_gliner(final_ds["train"])
val_simple = convert_to_gliner(final_ds["validation"])
test_simple = convert_to_gliner(final_ds["test"])

print(len(train_simple), len(val_simple), len(test_simple))
print(train_simple[0])


62097 3378 3415
{'tokenized_text': ['Heyvan', 'sahibinə', 'külli', 'miqdarda', 'maddi', 'ziyan', 'vurulub', '.'], 'ner': [[2, 3, 'number']]}


In [ ]:
for i in range(5):
    print("TEXT:", " ".join(train_simple[i]["tokenized_text"]))
    print("NER:", train_simple[i]["ner"])
    print("-" * 50)


TEXT: Heyvan sahibinə külli miqdarda maddi ziyan vurulub .
NER: [[2, 3, 'number']]
--------------------------------------------------
TEXT: Necə ki , Nuh peyğəmbərin oğlu öz atası və onun dininə etiqad etmədiyinə görə kafir kimi ölüb .
NER: [[3, 3, 'person']]
--------------------------------------------------
TEXT: Bununla belə , faiz qadağasını heç vaxt pozmamaq lazımdır .
NER: [[3, 3, 'number'], [6, 6, 'date']]
--------------------------------------------------
TEXT: Azərbaycanda il ərzində uşaqlara qarşı törədilən və müvafiq qaydada istintaqı tamamlanmış cinayətlərin ümumi sayı 500 fakt təşkil edir .
NER: [[0, 0, 'location'], [1, 1, 'date'], [14, 14, 'number']]
--------------------------------------------------
TEXT: Dedi ki , mülkiyyətçi sənsən , ancaq gəlin sənə qarşı iddia qaldırıb , sözün nədir , fikrin nədir .
NER: [[3, 3, 'position']]
--------------------------------------------------


## Label Simplification and Conversion to GLiNER Format

At this stage, the original label set was simplified in order to reduce annotation noise and make the training setup more stable. During the manual inspection of random samples, several labels appeared to be inconsistently applied or overly broad. To address this, only the more useful and relatively reliable categories were retained for training.

In addition, some semantically similar labels were merged into a single category. For example, `GPE` was mapped to `LOCATION`, and number-related categories such as `CARDINAL`, `ORDINAL`, `QUANTITY`, and `PERCENTAGE` were grouped under a single label, `NUMBER`. This was done to reduce ambiguity between closely related classes and to make the dataset easier for the model to learn from.

After simplifying the labels, the dataset was converted from token-level annotation format into the format expected by GLiNER. The original dataset stored annotations as `tokens` and `ner_tags`, where each token had a separate numeric tag. However, GLiNER expects entity annotations in span format.

The converted format has the following structure:

```python
{
    "tokenized_text": [...],
    "ner": [[start, end, label], ...]
}
```
In other words, the `start` and `end` indices allow the model to treat multiple consecutive tokens as a single entity. For example, if `["Ilham", "Aliyev"]` refers to one person, it should not be interpreted as two separate `person` entities. Instead, it is represented as `[0, 1, "person"]`, meaning that the entity starts at token index `0` and ends at token index `1`, and both tokens together form one complete `person` mention. Similarly, a single-token entity such as `["Bakida"]` would be represented as `[0, 0, "location"]`. This span-based representation helps the model learn that named entities may consist of one or multiple tokens and should be recognized as a unified semantic unit.


